In [0]:
# ============================================================
# 03_chunk_documents
# ============================================================
#
# Purpose:
#   Build a deterministic structure-aware chunking baseline
#   from cleaned World Bank GEP pages.
#
# Input:
#   worldbank_ai.silver.gep_clean_pages
#
# Outputs:
#   worldbank_ai.silver.gep_parent_chunks
#   worldbank_ai.silver.gep_child_chunks
#
# IMPORTANT:
#   This is the deterministic baseline.
#
#   More advanced strategies such as:
#       - recursive chunking
#       - parent-child variations
#       - LLM-assisted boundary detection
#
#   will be evaluated later in:
#       04_rag/01_chunking_experiments
# ============================================================

CATALOG = "worldbank_ai"
SCHEMA = "silver"

SOURCE_TABLE = (
    f"{CATALOG}.{SCHEMA}.gep_clean_pages"
)

PARENT_TABLE = (
    f"{CATALOG}.{SCHEMA}.gep_parent_chunks"
)

CHILD_TABLE = (
    f"{CATALOG}.{SCHEMA}.gep_child_chunks"
)


# ------------------------------------------------------------
# Baseline chunking configuration
# ------------------------------------------------------------

CHUNKING_METHOD = (
    "structure_aware_parent_child"
)

CHUNKING_VERSION = "baseline_v1"


# Child retrieval size.
#
# ~4 chars/token is a rough approximation.
# 2,500 chars ≈ 600 tokens.
TARGET_CHARS = 2500

MAX_CHARS = 3500

MIN_CHARS = 300

OVERLAP_CHARS = 300


print("Configuration loaded.")

print(f"Source: {SOURCE_TABLE}")
print(f"Parent target: {PARENT_TABLE}")
print(f"Child target:  {CHILD_TABLE}")

In [0]:
# ============================================================
# Imports
# ============================================================

import re
import math
import hashlib

from collections import Counter

from pyspark.sql import functions as F
from pyspark.sql import types as T


print("Imports loaded.")

In [0]:
# ============================================================
# Load cleaned pages
# ============================================================

clean_pages_df = (
    spark.table(SOURCE_TABLE)
)


page_count = (
    clean_pages_df.count()
)

document_count = (
    clean_pages_df
    .select("document_id")
    .distinct()
    .count()
)


print(
    f"Clean pages: {page_count:,}"
)

print(
    f"Documents:   {document_count:,}"
)


# ------------------------------------------------------------
# Validate expected corpus
# ------------------------------------------------------------

if page_count != 1098:

    raise RuntimeError(
        f"Expected 1,098 pages, "
        f"found {page_count:,}."
    )


if document_count != 5:

    raise RuntimeError(
        f"Expected 5 GEP documents, "
        f"found {document_count:,}."
    )


required_columns = [

    "document_id",
    "report_year",
    "page_number",
    "clean_text",
    "parsing_status",
    "image_count",
    "is_empty_after_cleaning"
]


missing_columns = [

    column
    for column in required_columns

    if column
    not in clean_pages_df.columns
]


if missing_columns:

    raise RuntimeError(
        f"Missing required columns: "
        f"{missing_columns}"
    )


print("Input validation passed.")

In [0]:
# ============================================================
# High-confidence GEP structural definitions
# ============================================================
#
# IMPORTANT:
#
# We intentionally DO NOT use generic title-case text as a
# structural boundary.
#
# PDF extraction contains strings such as:
#
#   World
#   China
#   Advanced Economies
#   Percent
#   Global Demand
#
# Some are chart labels, table labels or captions rather than
# actual section headings.
#
# Only high-confidence structure changes hierarchy here.
# ============================================================


KNOWN_SUBSECTIONS = {

    "recent developments":
        "Recent developments",

    "outlook":
        "Outlook",

    "risks":
        "Risks",

    "risks to the outlook":
        "Risks to the outlook",

    "policy challenges":
        "Policy challenges",

    "policy priorities":
        "Policy priorities",

    "policy implications":
        "Policy implications",

    "global outlook":
        "Global Outlook"
}


REGION_PATTERNS = {

    "East Asia and Pacific": [
        "east asia and pacific",
        "east asia & pacific"
    ],

    "Europe and Central Asia": [
        "europe and central asia",
        "europe & central asia"
    ],

    "Latin America and the Caribbean": [
        "latin america and the caribbean",
        "latin america & the caribbean"
    ],

    "Middle East and North Africa": [
        "middle east and north africa",
        "middle east & north africa"
    ],

    "South Asia": [
        "south asia"
    ],

    "Sub-Saharan Africa": [
        "sub-saharan africa",
        "sub saharan africa"
    ]
}


TABLE_PATTERN = re.compile(
    r"\bTABLE\s+[A-Z0-9.\-]+",
    re.IGNORECASE
)


FIGURE_PATTERN = re.compile(
    r"\bFIGURE\s+[A-Z0-9.\-]+",
    re.IGNORECASE
)


ROMAN_NUMERAL_PATTERN = re.compile(
    r"^[ivxlcdm]+$",
    re.IGNORECASE
)


print("Structural definitions loaded.")

In [0]:
# ============================================================
# Structural helper functions
# ============================================================


def normalize_heading(text):
    """
    Normalize a possible heading for comparison only.

    Original source text is never modified by this function.
    """

    if text is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        text.strip()
    )


# ============================================================
# Chapter detection
# ============================================================

def detect_chapter(line):

    text = normalize_heading(line)

    if not text.lower().startswith(
        "chapter"
    ):
        return None


    match = re.match(
        r"^chapter\s+(\d+)"
        r"(?:\s*[:.\-]?\s*(.*))?$",
        text,
        flags=re.IGNORECASE
    )


    if not match:
        return None


    return {

        "chapter_number":
            match.group(1),

        "chapter_title":
            (
                match.group(2).strip()
                if match.group(2)
                else None
            )
    }


# ============================================================
# Known subsection detection
# ============================================================

def detect_known_subsection(line):

    text = (
        normalize_heading(line)
        .lower()
        .rstrip(":")
        .strip()
    )


    return KNOWN_SUBSECTIONS.get(
        text
    )


# ============================================================
# Region detection
# ============================================================

def detect_region(line):

    text = (
        normalize_heading(line)
        .lower()
        .rstrip(":")
        .strip()
    )


    for region, patterns in (
        REGION_PATTERNS.items()
    ):

        if text in patterns:
            return region


    return None


# ============================================================
# Running-header detection
# ============================================================

def normalize_header(text):

    if not text:
        return ""

    return re.sub(
        r"[^A-Z0-9]",
        "",
        text.upper()
    )


def is_running_header(text):

    normalized = normalize_header(
        text
    )


    if (
        "GLOBALECONOMICPROSPECTS"
        in normalized
        and
        "JANUARY"
        in normalized
    ):
        return True


    return False


# ============================================================
# Safe noise detection
# ============================================================

def is_safe_noise_block(text):

    if not text:
        return True


    text = text.strip()


    if not text:
        return True


    # Remove Roman page numbers such as:
    # vii, ix, xvii, xix
    if (
        len(text) <= 8
        and
        ROMAN_NUMERAL_PATTERN.fullmatch(
            text
        )
    ):
        return True


    return False


# ============================================================
# Table / figure classification
# ============================================================

def is_table_candidate(text):

    if not text:
        return False

    return bool(
        TABLE_PATTERN.search(text)
    )


def is_figure_candidate(text):

    if not text:
        return False

    return bool(
        FIGURE_PATTERN.search(text)
    )


print("Structural helpers loaded.")

In [0]:
# ============================================================
# Table-of-contents detection
# ============================================================


def is_toc_like_page(text):

    if not text:
        return False


    lines = [

        line.strip().lower()

        for line in text.splitlines()

        if line.strip()
    ]


    # --------------------------------------------------------
    # Explicit TOC heading
    # --------------------------------------------------------

    toc_headings = {

        "contents",
        "table of contents",
        "summary of contents"
    }


    if any(
        line in toc_headings
        for line in lines[:15]
    ):
        return True


    # --------------------------------------------------------
    # Typical dotted TOC entries
    #
    # Example:
    # Global Outlook ................. 3
    # --------------------------------------------------------

    dotted_entries = len(

        re.findall(
            r"\.{4,}\s*\d+",
            text
        )
    )


    structural_terms = [

        "outlook",
        "risks",
        "recent developments",
        "south asia",
        "sub-saharan africa",
        "east asia",
        "europe and central asia"
    ]


    term_count = sum(

        1

        for term in structural_terms

        if term in text.lower()
    )


    if (
        dotted_entries >= 3
        and
        term_count >= 3
    ):
        return True


    return False


print("TOC detector loaded.")

In [0]:
# ============================================================
# Collect pages in deterministic order
# ============================================================
#
# Structural parsing is stateful across pages.
#
# We only have 1,098 pages, so driver-side parsing is
# reasonable for this controlled corpus.
# ============================================================


page_rows = (

    clean_pages_df

    .select(
        "document_id",
        "report_year",
        "page_number",
        "clean_text",
        "parsing_status",
        "image_count"
    )

    .orderBy(
        "document_id",
        "page_number"
    )

    .collect()
)


print(
    f"Collected {len(page_rows):,} pages."
)

In [0]:
# ============================================================
# Build deterministic structure-aware blocks
# ============================================================

structural_blocks = []

block_sequence = 0


# ------------------------------------------------------------
# Current document hierarchy
# ------------------------------------------------------------

current_document_id = None
current_report_year = None

current_chapter = None
current_chapter_title = None

current_region = None
current_section = None
current_subsection = None


# ------------------------------------------------------------
# Current text buffer
# ------------------------------------------------------------

buffer_lines = []

buffer_page_start = None
buffer_page_end = None


# ============================================================
# Flush current text buffer
# ============================================================

def flush_buffer():

    global block_sequence

    global buffer_lines
    global buffer_page_start
    global buffer_page_end


    if not buffer_lines:
        return


    block_text = (
        "\n".join(buffer_lines)
        .strip()
    )


    if is_safe_noise_block(
        block_text
    ):

        buffer_lines = []
        buffer_page_start = None
        buffer_page_end = None

        return


    if is_table_candidate(
        block_text
    ):

        content_type = (
            "table_candidate"
        )


    elif is_figure_candidate(
        block_text
    ):

        content_type = (
            "figure_candidate"
        )


    else:

        content_type = "narrative"


    block_sequence += 1


    structural_blocks.append({

        "document_id":
            current_document_id,

        "report_year":
            current_report_year,

        "chapter":
            current_chapter,

        "chapter_title":
            current_chapter_title,

        "region":
            current_region,

        "section":
            current_section,

        "subsection":
            current_subsection,

        "content_type":
            content_type,

        "page_start":
            buffer_page_start,

        "page_end":
            buffer_page_end,

        "block_sequence":
            block_sequence,

        "block_text":
            block_text,

        "character_count":
            len(block_text)
    })


    buffer_lines = []
    buffer_page_start = None
    buffer_page_end = None


# ============================================================
# Process pages
# ============================================================

for row in page_rows:

    document_id = (
        row["document_id"]
    )

    report_year = (
        row["report_year"]
    )

    page_number = (
        row["page_number"]
    )

    page_text = (
        row["clean_text"]
        or ""
    )


    # ========================================================
    # New document
    # ========================================================

    if (
        document_id
        != current_document_id
    ):

        flush_buffer()


        current_document_id = (
            document_id
        )

        current_report_year = (
            report_year
        )


        current_chapter = None
        current_chapter_title = None

        current_region = None
        current_section = None
        current_subsection = None


    # ========================================================
    # TOC page
    # ========================================================

    if is_toc_like_page(
        page_text
    ):

        flush_buffer()


        if page_text.strip():

            block_sequence += 1


            structural_blocks.append({

                "document_id":
                    document_id,

                "report_year":
                    report_year,

                "chapter":
                    None,

                "chapter_title":
                    None,

                "region":
                    None,

                "section":
                    None,

                "subsection":
                    None,

                "content_type":
                    "front_matter",

                "page_start":
                    page_number,

                "page_end":
                    page_number,

                "block_sequence":
                    block_sequence,

                "block_text":
                    page_text.strip(),

                "character_count":
                    len(
                        page_text.strip()
                    )
            })


        continue


    # ========================================================
    # Process page lines
    # ========================================================

    for raw_line in (
        page_text.splitlines()
    ):

        line = raw_line.strip()


        if not line:
            continue


        # ----------------------------------------------------
        # Ignore running headers.
        # ----------------------------------------------------

        if is_running_header(
            line
        ):
            continue


        # ----------------------------------------------------
        # Ignore safe page-number noise.
        # ----------------------------------------------------

        if is_safe_noise_block(
            line
        ):
            continue


        # ====================================================
        # Chapter
        # ====================================================

        chapter_info = (
            detect_chapter(line)
        )


        if chapter_info:

            flush_buffer()


            new_chapter = (
                chapter_info[
                    "chapter_number"
                ]
            )


            # Only reset lower hierarchy when
            # the chapter actually changes.
            if (
                new_chapter
                != current_chapter
            ):

                current_region = None
                current_section = None
                current_subsection = None


            current_chapter = (
                new_chapter
            )

            current_chapter_title = (
                chapter_info[
                    "chapter_title"
                ]
            )


            continue


        # ====================================================
        # Region
        # ====================================================

        region = detect_region(
            line
        )


        if region:

            flush_buffer()


            if (
                region
                != current_region
            ):

                current_section = None
                current_subsection = None


            current_region = region

            continue


        # ====================================================
        # Known subsection
        # ====================================================

        subsection = (
            detect_known_subsection(
                line
            )
        )


        if subsection:

            flush_buffer()


            current_section = (
                subsection
            )

            current_subsection = (
                subsection
            )


            continue


        # ====================================================
        # Ordinary source text
        # ====================================================

        if buffer_page_start is None:

            buffer_page_start = (
                page_number
            )


        buffer_page_end = (
            page_number
        )


        buffer_lines.append(
            line
        )


# Flush final document.
flush_buffer()


print(
    f"Structural blocks created: "
    f"{len(structural_blocks):,}"
)

In [0]:
# ============================================================
# Structural block validation
# ============================================================


if not structural_blocks:

    raise RuntimeError(
        "No structural blocks were created."
    )


print(
    f"Structural blocks: "
    f"{len(structural_blocks):,}"
)


print(
    "Documents:",
    len({
        block["document_id"]
        for block in structural_blocks
    })
)


print(
    "Report years:",
    sorted({
        block["report_year"]
        for block in structural_blocks
    })
)


# ------------------------------------------------------------
# Content-type distribution
# ------------------------------------------------------------

content_counts = Counter(

    block["content_type"]

    for block in structural_blocks
)


print("\nContent types:")


for content_type, count in (
    content_counts.items()
):

    print(
        f"  {content_type}: "
        f"{count:,}"
    )

In [0]:
# ============================================================
# Build logical parent chunks
# ============================================================
#
# A parent represents a coherent structural section.
#
# Adjacent blocks are merged only when their structural
# metadata matches.
#
# We do NOT force tiny parents to merge across semantic
# boundaries in this baseline.
# ============================================================


def structural_key(block):

    return (

        block["document_id"],
        block["chapter"],
        block["chapter_title"],
        block["region"],
        block["section"],
        block["subsection"],
        block["content_type"]
    )


parent_records = []

current_parent = None


for block in structural_blocks:

    key = structural_key(
        block
    )


    # --------------------------------------------------------
    # Start first parent.
    # --------------------------------------------------------

    if current_parent is None:

        current_parent = {

            "key":
                key,

            "document_id":
                block["document_id"],

            "report_year":
                block["report_year"],

            "chapter":
                block["chapter"],

            "chapter_title":
                block["chapter_title"],

            "region":
                block["region"],

            "section":
                block["section"],

            "subsection":
                block["subsection"],

            "content_type":
                block["content_type"],

            "page_start":
                block["page_start"],

            "page_end":
                block["page_end"],

            "texts": [
                block["block_text"]
            ]
        }


        continue


    # --------------------------------------------------------
    # Same structural context -> merge.
    # --------------------------------------------------------

    if (
        key
        == current_parent["key"]
    ):

        current_parent[
            "texts"
        ].append(
            block["block_text"]
        )


        current_parent[
            "page_end"
        ] = max(

            current_parent[
                "page_end"
            ],

            block["page_end"]
        )


    else:

        parent_records.append(
            current_parent
        )


        current_parent = {

            "key":
                key,

            "document_id":
                block["document_id"],

            "report_year":
                block["report_year"],

            "chapter":
                block["chapter"],

            "chapter_title":
                block["chapter_title"],

            "region":
                block["region"],

            "section":
                block["section"],

            "subsection":
                block["subsection"],

            "content_type":
                block["content_type"],

            "page_start":
                block["page_start"],

            "page_end":
                block["page_end"],

            "texts": [
                block["block_text"]
            ]
        }


# Flush final parent.
if current_parent:

    parent_records.append(
        current_parent
    )


print(
    f"Logical parent sections: "
    f"{len(parent_records):,}"
)

In [0]:
# ============================================================
# Finalize parent chunks
# ============================================================


def stable_hash(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()[:20]


final_parent_records = []


for parent_sequence, parent in enumerate(
    parent_records,
    start=1
):

    parent_text = (
        "\n\n".join(
            parent["texts"]
        )
        .strip()
    )


    if not parent_text:
        continue


    parent_identity = "|".join([

        str(
            parent["document_id"]
        ),

        str(
            parent["chapter"]
        ),

        str(
            parent["region"]
        ),

        str(
            parent["section"]
        ),

        str(
            parent["subsection"]
        ),

        str(
            parent["content_type"]
        ),

        str(
            parent["page_start"]
        ),

        str(
            parent["page_end"]
        ),

        str(
            parent_sequence
        )
    ])


    parent_chunk_id = (
        "parent_"
        + stable_hash(
            parent_identity
        )
    )


    final_parent_records.append({

        "parent_chunk_id":
            parent_chunk_id,

        "document_id":
            parent["document_id"],

        "report_year":
            int(
                parent["report_year"]
            ),

        "chapter":
            parent["chapter"],

        "chapter_title":
            parent["chapter_title"],

        "section":
            parent["section"],

        "subsection":
            parent["subsection"],

        "region":
            parent["region"],

        "content_type":
            parent["content_type"],

        "page_start":
            int(
                parent["page_start"]
            ),

        "page_end":
            int(
                parent["page_end"]
            ),

        "parent_sequence":
            parent_sequence,

        "parent_text":
            parent_text,

        "character_count":
            len(parent_text),

        "approx_token_count":
            math.ceil(
                len(parent_text) / 4
            ),

        "chunking_method":
            CHUNKING_METHOD,

        "chunking_version":
            CHUNKING_VERSION
    })


print(
    f"Final parent chunks: "
    f"{len(final_parent_records):,}"
)

In [0]:
# ============================================================
# Child retrieval chunking helpers
# ============================================================


def split_into_units(text):
    """
    Split parent text primarily by paragraphs.

    If extraction has no paragraph boundaries, fall back
    to sentence boundaries.
    """

    if not text:
        return []


    paragraphs = [

        part.strip()

        for part in re.split(
            r"\n{2,}",
            text
        )

        if part.strip()
    ]


    # Good paragraph structure exists.
    if len(paragraphs) > 1:
        return paragraphs


    # --------------------------------------------------------
    # Sentence fallback
    # --------------------------------------------------------

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )


    return [

        sentence.strip()

        for sentence in sentences

        if sentence.strip()
    ]


def create_overlap(
    text,
    overlap_chars
):
    """
    Return trailing semantic overlap.
    """

    if not text:
        return ""


    if len(text) <= overlap_chars:

        return text


    overlap = (
        text[-overlap_chars:]
    )


    # Avoid starting in the middle of a word.
    first_space = (
        overlap.find(" ")
    )


    if first_space != -1:

        overlap = (
            overlap[
                first_space + 1:
            ]
        )


    return overlap.strip()

In [0]:
# ============================================================
# Create child retrieval chunks
# ============================================================


def create_child_texts(parent_text):

    if not parent_text:
        return []


    parent_text = (
        parent_text.strip()
    )


    # --------------------------------------------------------
    # Small parent does not need additional fragmentation.
    # --------------------------------------------------------

    if len(parent_text) <= MAX_CHARS:

        return [
            parent_text
        ]


    units = split_into_units(
        parent_text
    )


    chunks = []

    current_parts = []
    current_length = 0


    for unit in units:

        unit = unit.strip()


        if not unit:
            continue


        # ====================================================
        # Oversized single unit
        # ====================================================

        if len(unit) > MAX_CHARS:

            # Flush current content first.
            if current_parts:

                completed = (
                    "\n\n".join(
                        current_parts
                    )
                    .strip()
                )


                if completed:

                    chunks.append(
                        completed
                    )


                current_parts = []
                current_length = 0


            # -----------------------------------------------
            # Hard split only as a fallback.
            # -----------------------------------------------

            start = 0


            while start < len(unit):

                end = min(
                    start + MAX_CHARS,
                    len(unit)
                )


                # Try to avoid cutting a word.
                if end < len(unit):

                    split_position = (
                        unit.rfind(
                            " ",
                            start,
                            end
                        )
                    )


                    if (
                        split_position
                        > start
                    ):

                        end = (
                            split_position
                        )


                piece = (
                    unit[start:end]
                    .strip()
                )


                if piece:

                    chunks.append(
                        piece
                    )


                if end >= len(unit):
                    break


                start = max(
                    end - OVERLAP_CHARS,
                    start + 1
                )


            continue


        # ====================================================
        # Normal semantic unit
        # ====================================================

        projected_length = (

            current_length

            + (
                2
                if current_parts
                else 0
            )

            + len(unit)
        )


        if (
            current_parts
            and
            projected_length
            > TARGET_CHARS
        ):

            completed = (
                "\n\n".join(
                    current_parts
                )
                .strip()
            )


            if completed:

                chunks.append(
                    completed
                )


            overlap = create_overlap(
                completed,
                OVERLAP_CHARS
            )


            current_parts = []


            if overlap:

                current_parts.append(
                    overlap
                )


            current_parts.append(
                unit
            )


            current_length = len(

                "\n\n".join(
                    current_parts
                )
            )


        else:

            current_parts.append(
                unit
            )

            current_length = (
                projected_length
            )


    # ========================================================
    # Flush final chunk
    # ========================================================

    if current_parts:

        final_text = (
            "\n\n".join(
                current_parts
            )
            .strip()
        )


        if final_text:

            chunks.append(
                final_text
            )


    # ========================================================
    # Merge tiny trailing fragment when safe
    # ========================================================

    if (
        len(chunks) >= 2
        and
        len(chunks[-1]) < MIN_CHARS
    ):

        merged = (

            chunks[-2].rstrip()

            + "\n\n"

            + chunks[-1].lstrip()
        )


        if len(merged) <= MAX_CHARS:

            chunks[-2] = merged

            chunks.pop()


    return chunks


print(
    "Child chunking helper loaded."
)

In [0]:
# ============================================================
# Parent DataFrame
# ============================================================


parent_schema = T.StructType([

    T.StructField(
        "parent_chunk_id",
        T.StringType(),
        False
    ),

    T.StructField(
        "document_id",
        T.StringType(),
        False
    ),

    T.StructField(
        "report_year",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "chapter",
        T.StringType(),
        True
    ),

    T.StructField(
        "chapter_title",
        T.StringType(),
        True
    ),

    T.StructField(
        "section",
        T.StringType(),
        True
    ),

    T.StructField(
        "subsection",
        T.StringType(),
        True
    ),

    T.StructField(
        "region",
        T.StringType(),
        True
    ),

    T.StructField(
        "content_type",
        T.StringType(),
        False
    ),

    T.StructField(
        "page_start",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "page_end",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "parent_sequence",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "parent_text",
        T.StringType(),
        False
    ),

    T.StructField(
        "character_count",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "approx_token_count",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "chunking_method",
        T.StringType(),
        False
    ),

    T.StructField(
        "chunking_version",
        T.StringType(),
        False
    )
])


parent_df = (

    spark.createDataFrame(
        final_parent_records,
        schema=parent_schema
    )

    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)


print(
    f"Parent chunks: "
    f"{parent_df.count():,}"
)

In [0]:
# ============================================================
# Generate child retrieval chunks
# ============================================================


child_records = []


for parent in final_parent_records:

    child_texts = (
        create_child_texts(
            parent["parent_text"]
        )
    )


    for child_index, child_text in enumerate(
        child_texts,
        start=1
    ):

        identity = "|".join([

            parent[
                "parent_chunk_id"
            ],

            str(child_index),

            child_text
        ])


        chunk_id = (
            "chunk_"
            + stable_hash(identity)
        )


        child_records.append({

            "chunk_id":
                chunk_id,

            "parent_chunk_id":
                parent[
                    "parent_chunk_id"
                ],

            "document_id":
                parent[
                    "document_id"
                ],

            "report_year":
                parent[
                    "report_year"
                ],

            "chapter":
                parent["chapter"],

            "chapter_title":
                parent[
                    "chapter_title"
                ],

            "section":
                parent["section"],

            "subsection":
                parent[
                    "subsection"
                ],

            "region":
                parent["region"],

            "content_type":
                parent[
                    "content_type"
                ],

            # Conservative page provenance for baseline.
            "page_start":
                parent[
                    "page_start"
                ],

            "page_end":
                parent[
                    "page_end"
                ],

            "chunk_index":
                child_index,

            "chunk_text":
                child_text,

            "character_count":
                len(child_text),

            "approx_token_count":
                math.ceil(
                    len(child_text) / 4
                ),

            "chunking_method":
                CHUNKING_METHOD,

            "chunking_version":
                CHUNKING_VERSION
        })


print(
    f"Child chunks created: "
    f"{len(child_records):,}"
)

In [0]:
# ============================================================
# Child DataFrame
# ============================================================


child_schema = T.StructType([

    T.StructField(
        "chunk_id",
        T.StringType(),
        False
    ),

    T.StructField(
        "parent_chunk_id",
        T.StringType(),
        False
    ),

    T.StructField(
        "document_id",
        T.StringType(),
        False
    ),

    T.StructField(
        "report_year",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "chapter",
        T.StringType(),
        True
    ),

    T.StructField(
        "chapter_title",
        T.StringType(),
        True
    ),

    T.StructField(
        "section",
        T.StringType(),
        True
    ),

    T.StructField(
        "subsection",
        T.StringType(),
        True
    ),

    T.StructField(
        "region",
        T.StringType(),
        True
    ),

    T.StructField(
        "content_type",
        T.StringType(),
        False
    ),

    T.StructField(
        "page_start",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "page_end",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "chunk_index",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "chunk_text",
        T.StringType(),
        False
    ),

    T.StructField(
        "character_count",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "approx_token_count",
        T.IntegerType(),
        False
    ),

    T.StructField(
        "chunking_method",
        T.StringType(),
        False
    ),

    T.StructField(
        "chunking_version",
        T.StringType(),
        False
    )
])


child_df = (

    spark.createDataFrame(
        child_records,
        schema=child_schema
    )

    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)


print(
    f"Parent chunks: "
    f"{parent_df.count():,}"
)

print(
    f"Child chunks:  "
    f"{child_df.count():,}"
)

print(
    "Parent and child DataFrames are ready."
)

In [0]:
# ============================================================
# Final baseline validation
# ============================================================


parent_count = (
    parent_df.count()
)

child_count = (
    child_df.count()
)


# ------------------------------------------------------------
# ID uniqueness
# ------------------------------------------------------------

unique_parent_count = (

    parent_df

    .select(
        "parent_chunk_id"
    )

    .distinct()

    .count()
)


unique_child_count = (

    child_df

    .select(
        "chunk_id"
    )

    .distinct()

    .count()
)


if (
    unique_parent_count
    != parent_count
):

    raise RuntimeError(
        "Duplicate parent IDs detected."
    )


if (
    unique_child_count
    != child_count
):

    raise RuntimeError(
        "Duplicate child IDs detected."
    )


# ------------------------------------------------------------
# Referential integrity
# ------------------------------------------------------------

orphan_count = (

    child_df.alias("c")

    .join(

        parent_df
        .select(
            "parent_chunk_id"
        )
        .alias("p"),

        on="parent_chunk_id",

        how="left_anti"
    )

    .count()
)


if orphan_count != 0:

    raise RuntimeError(
        f"Found {orphan_count} "
        "orphan child chunks."
    )


# ------------------------------------------------------------
# Size metrics
# ------------------------------------------------------------

tiny_count = (

    child_df

    .filter(
        F.col(
            "character_count"
        ) < MIN_CHARS
    )

    .count()
)


oversized_count = (

    child_df

    .filter(
        F.col(
            "character_count"
        ) > MAX_CHARS
    )

    .count()
)


cross_page_count = (

    parent_df

    .filter(
        F.col("page_end")
        >
        F.col("page_start")
    )

    .count()
)


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

size_stats = (

    child_df

    .agg(

        F.round(
            F.avg(
                "character_count"
            ),
            2
        ).alias(
            "avg_chars"
        ),

        F.min(
            "character_count"
        ).alias(
            "min_chars"
        ),

        F.expr(
            "percentile_approx("
            "character_count, 0.50)"
        ).alias(
            "median_chars"
        ),

        F.expr(
            "percentile_approx("
            "character_count, 0.90)"
        ).alias(
            "p90_chars"
        ),

        F.expr(
            "percentile_approx("
            "character_count, 0.95)"
        ).alias(
            "p95_chars"
        ),

        F.max(
            "character_count"
        ).alias(
            "max_chars"
        )
    )

    .first()
)


print("=" * 70)

print(
    "DETERMINISTIC CHUNKING BASELINE"
)

print("=" * 70)


print(
    f"Method:                  "
    f"{CHUNKING_METHOD}"
)

print(
    f"Version:                 "
    f"{CHUNKING_VERSION}"
)

print()

print(
    f"Parent chunks:           "
    f"{parent_count:,}"
)

print(
    f"Child chunks:            "
    f"{child_count:,}"
)

print(
    f"Children / parent:       "
    f"{child_count / parent_count:.2f}"
)

print()

print(
    f"Average chars:           "
    f"{size_stats['avg_chars']}"
)

print(
    f"Minimum chars:           "
    f"{size_stats['min_chars']}"
)

print(
    f"Median chars:            "
    f"{size_stats['median_chars']}"
)

print(
    f"P90 chars:               "
    f"{size_stats['p90_chars']}"
)

print(
    f"P95 chars:               "
    f"{size_stats['p95_chars']}"
)

print(
    f"Maximum chars:           "
    f"{size_stats['max_chars']}"
)

print()

print(
    f"Children < {MIN_CHARS}:          "
    f"{tiny_count:,}"
)

print(
    f"Children > {MAX_CHARS}:         "
    f"{oversized_count:,}"
)

print(
    f"Cross-page parents:      "
    f"{cross_page_count:,}"
)

print(
    f"Orphan children:         "
    f"{orphan_count:,}"
)

print()

print(
    f"Unique parent IDs:       "
    f"{unique_parent_count:,}"
)

print(
    f"Unique child IDs:        "
    f"{unique_child_count:,}"
)

print("=" * 70)

In [0]:
# ============================================================
# Inspect remaining small chunks
# ============================================================


tiny_chunks_df = (

    child_df

    .filter(
        F.col(
            "character_count"
        ) < MIN_CHARS
    )

    .select(

        "report_year",

        "chapter",
        "chapter_title",

        "region",

        "section",
        "subsection",

        "content_type",

        "page_start",
        "page_end",

        "character_count",

        "chunk_text"
    )

    .orderBy(
        "character_count"
    )
)


print(
    f"Small baseline chunks: "
    f"{tiny_chunks_df.count():,}"
)


display(
    tiny_chunks_df.limit(50)
)

In [0]:
# ============================================================
# Content-type distribution
# ============================================================


display(

    child_df

    .groupBy(
        "report_year",
        "content_type"
    )

    .count()

    .orderBy(
        "report_year",
        "content_type"
    )
)

In [0]:
# ============================================================
# Important RAG structure sanity check
# ============================================================


display(

    child_df

    .filter(

        (
            F.col("region")
            == "South Asia"
        )

        |

        (
            F.lower(
                F.coalesce(
                    F.col("subsection"),
                    F.lit("")
                )
            )
            .contains("risk")
        )
    )

    .select(

        "report_year",

        "chapter",

        "region",

        "section",
        "subsection",

        "page_start",
        "page_end",

        "character_count",

        "chunk_text"
    )

    .orderBy(
        "report_year",
        "page_start"
    )

    .limit(100)
)

In [0]:
# ============================================================
# Persist deterministic baseline
# ============================================================

(
    parent_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PARENT_TABLE)
)

(
    child_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHILD_TABLE)
)

print(f"Saved parent baseline: {PARENT_TABLE}")
print(f"Saved child baseline:  {CHILD_TABLE}")

In [0]:
# ============================================================
# Final persisted-table validation
# ============================================================

saved_parent_df = spark.table(PARENT_TABLE)
saved_child_df = spark.table(CHILD_TABLE)

saved_parent_count = saved_parent_df.count()
saved_child_count = saved_child_df.count()

assert saved_parent_count == parent_df.count(), \
    "Persisted parent count mismatch."

assert saved_child_count == child_df.count(), \
    "Persisted child count mismatch."

print("=" * 70)
print("03_chunk_documents COMPLETED")
print("=" * 70)
print(f"Saved parents:  {saved_parent_count:,}")
print(f"Saved children: {saved_child_count:,}")
print("=" * 70)